# INTERLEAVED generic-block solver — unlocking HARD_BENCH block-by-block

HARD_BENCH is calibrated so a frozen Qwen-1.5B scores ~0% because each task needs a **loop + a mutable register** a single forward pass lacks (a running count, a partial-product accumulator, a remainder, the last terms of a recurrence).

**This is the scratchpad fix as a library of REUSABLE GENERIC plan-blocks** (no problem-specific words). `interleaved_solver.py` runs them interleaved: the scratchpad starts as the problem; each generic block is fed the scratchpad-so-far, the model executes only that step and writes its intermediate, which is re-inserted for the next block, until a `FINAL ANSWER:` emerges. The generated text becomes the register; per-line emission becomes the loop body.

10 reusable skills cover all 20 categories (`enumerate_and_tally` -> all counting; `decompose_and_recombine` -> multiplication & modular-exp). The harness does a per-category variant search + generic mutation toward 100%, and NEVER injects a problem-specific block (a genericness gate enforces it). Watch accuracy climb from ~0% to (hopefully) 100% and read the final generic plan that solved each category.

## 0 GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU (CPU slow). Colab: Runtime -> GPU.'

## 1 Clone & install (latest commit)

In [ ]:
import os
!rm -rf small_fable
!git clone -q https://github.com/sinha-k-prat/small_fable.git
%cd small_fable
!git log --oneline -1
!pip install -q -r requirements.txt
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
print('Ready.')

## 2 Build HARD_BENCH + show the reusable generic skills

In [ ]:
!python -u tools/gen_hard_bench.py --seed 20240621 --out hard_bench.jsonl
import tools.interleaved_blocks as IB
print('skills:', len(IB.SKILLS), '| categories covered:', len(IB.CATEGORY_SKILL))
for name, sk in IB.SKILLS.items():
    print(f"  {name:24} -> {', '.join(sk['categories'][:4])}")
print('\nexample skill decompose_and_recombine, variant 0:')
for b in IB.SKILLS['decompose_and_recombine']['variants'][0]:
    print('   - ' + b)

## 3 Run the interleaved solver (per-category variant search + generic mutation toward 100%)
Many decodes (blocks x variants x rows). Start with --limit 3, then full. A100 -> bf16; T4 -> fp16.

In [ ]:
DTYPE='fp16'  # 'bf16' on A100
# start with --limit 3 per category to sanity-check fast, then drop --limit for the full run
!python -u interleaved_solver.py --data hard_bench.jsonl --base Qwen/Qwen2.5-1.5B-Instruct \
    --dtype $DTYPE --device cuda --limit 3 --max_new_tokens 320 --rounds 4 --out interleaved_out

## 4 Per-category accuracy + the FINAL generic plan per category

In [ ]:
import json, os
from IPython.display import Image, display
p = 'interleaved_out/results.json'
if not os.path.exists(p):
    print('[!] run cell 3 first')
else:
    res = json.load(open(p))
    png = 'interleaved_out/interleaved.png'
    if os.path.exists(png):
        display(Image(filename=png))
    print('per-category accuracy:')
    for c, a in sorted(res.get('per_category_accuracy', {}).items()):
        print(f'  {c:28} {a:.0%}')
    print('\nFINAL generic interleaved plan that solved each category:')
    for c, plan in res.get('final_generic_plans', {}).items():
        acc = plan.get('accuracy','') if isinstance(plan, dict) else ''
        print(f'\n[{c}]  acc={acc}')
        for b in (plan.get('blocks', plan) if isinstance(plan, dict) else plan):
            print('   - ' + str(b))